# Deep CFR GPU Advantage-Network Architecture Screen

This experiment changes only the advantage-network architecture. The average-strategy networks, data-generation budget, optimizer settings, and exact evaluation procedure remain fixed.

Primary metrics are exact generated-average exploitability and its normalized area under the curve versus measured training time. Exact evaluation and dense averaging are excluded from the training-time budget.

In [ ]:
import gc
import sys
import time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from liars_poker.algo.deep_cfr import DeepCFRTrainer
from liars_poker.algo.deep_cfr_diagnostics import ExactDenseStrategyAverager
from liars_poker.core import GameSpec
from liars_poker.policies.neural import NeuralMLP
from liars_poker.training.deep_cfr import _exact_evaluation

assert torch.cuda.is_available(), 'This notebook is intended for a CUDA GPU.'
device = torch.device('cuda')
torch.set_float32_matmul_precision('high')
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=3,
    claim_kinds=('RankHigh', 'Pair', 'Trips'),
    suit_symmetry=True,
)

SCREEN_MINUTES_PER_ARCHITECTURE = 5
EVALUATE_EVERY_TRAINING_SECONDS = 60
SEED = 7

# Keep these fixed while screening architecture.
STRATEGY_HIDDEN_SIZES = (512, 512)
TRAVERSALS_PER_PLAYER = 1024
TRAVERSAL_BATCH_SIZE = 1024
FITTING_BATCH_SIZE = 4096
ADVANTAGE_TRAIN_STEPS = 100
STRATEGY_TRAIN_STEPS = 50

# One means a true per-iteration exact generated average. Increasing this is
# faster in real wall time, but then the result is only a sampled average.
EXACT_AVERAGE_EVERY = 1

ADVANTAGE_ARCHITECTURES = {
    '512x512 control': (512, 512),
    '1024x1024': (1024, 1024),
    '2048x2048': (2048, 2048),
    '512x512x512x512': (512, 512, 512, 512),
    '1024x1024x1024x1024': (1024, 1024, 1024, 1024),
}

spec.to_short_str(), ADVANTAGE_ARCHITECTURES

## Experiment helpers

`DeepCFRTrainer.hidden_sizes` currently configures both network families. The helper below constructs the trainer with fixed `512x512` strategy networks, then replaces only its advantage networks and optimizers. These experimental trainers should not be checkpointed because the production checkpoint metadata currently records only one shared architecture.

In [ ]:
def make_architecture_trainer(hidden_sizes, *, seed):
    trainer = DeepCFRTrainer(
        spec,
        hidden_sizes=STRATEGY_HIDDEN_SIZES,
        device=device,
        seed=seed,
        advantage_buffer_capacity=1_000_000,
        strategy_buffer_capacity=1_000_000,
        learning_rate=1e-3,
        batch_size=FITTING_BATCH_SIZE,
        advantage_train_steps=ADVANTAGE_TRAIN_STEPS,
        strategy_train_steps=STRATEGY_TRAIN_STEPS,
        advantage_positive_weight=0.5,
        strategy_weighting='linear',
        highest_regret_fallback=True,
        alternating_updates=True,
        validation_fraction=0.02,
        validation_buffer_capacity=20_000,
        traversal_backend='gpu_native',
        traversal_batch_size=TRAVERSAL_BATCH_SIZE,
        device_replay=True,
        fused_optimizer=True,
        amp_dtype=None,
        compile_models=False,
    )

    torch.manual_seed(seed + 10_000)
    trainer.advantage_nets = [
        NeuralMLP(trainer.encoder.input_dim, trainer.encoder.action_dim, hidden_sizes).to(device)
        for _ in range(2)
    ]
    trainer.advantage_optimizers = [
        trainer._make_optimizer(model) for model in trainer.advantage_nets
    ]
    trainer._compiled_forwards.clear()
    return trainer


def parameter_count(model):
    return sum(parameter.numel() for parameter in model.parameters())


def observe_exact_average(trainer, averager):
    if trainer.iteration % EXACT_AVERAGE_EVERY != 0:
        return
    weight = 1.0 if trainer.strategy_weighting == 'uniform' else float(trainer.iteration + 1)
    averager.observe(trainer.current_policy_dense(), weight=weight)


def train_for_fixed_time(label, hidden_sizes, *, seed, training_seconds):
    trainer = make_architecture_trainer(hidden_sizes, seed=seed)
    averager = ExactDenseStrategyAverager(spec)
    training_records = []
    evaluations = []
    measured_training_s = 0.0
    next_evaluation_s = float(EVALUATE_EVERY_TRAINING_SECONDS)
    actual_start = time.perf_counter()

    while measured_training_s < training_seconds:
        observe_exact_average(trainer, averager)
        start = time.perf_counter()
        record = trainer.run_iteration(traversals_per_player=TRAVERSALS_PER_PLAYER)
        measured_training_s += time.perf_counter() - start
        record['measured_training_s'] = measured_training_s
        training_records.append(record)

        if measured_training_s >= next_evaluation_s or measured_training_s >= training_seconds:
            validation = trainer.validation_metrics()
            metrics = _exact_evaluation(spec, trainer, averager)
            evaluations.append({
                'configuration': label,
                'iteration': trainer.iteration,
                'measured_training_s': measured_training_s,
                'actual_wall_s': time.perf_counter() - actual_start,
                'advantage_validation_mse': float(np.mean([
                    player['mse'] for player in validation['advantage']
                ])),
                'advantage_validation_strategy_tv': float(np.mean([
                    player['strategy_tv'] for player in validation['advantage']
                ])),
                **metrics,
            })
            print(
                f"[{label}] train={measured_training_s / 60:.1f}m "
                f"iter={trainer.iteration} "
                f"exact_avg={2 * metrics['exact_average_predicted_avg'] - 1:.5f}"
            )
            while next_evaluation_s <= measured_training_s:
                next_evaluation_s += EVALUATE_EVERY_TRAINING_SECONDS

    result = {
        'configuration': label,
        'hidden_sizes': tuple(hidden_sizes),
        'parameters_per_advantage_network': parameter_count(trainer.advantage_nets[0]),
        'parameters_per_strategy_network': parameter_count(trainer.strategy_nets[0]),
        'iterations': trainer.iteration,
        'training_records': training_records,
        'evaluations': evaluations,
    }
    return result

## Run the equal-time architecture screen

This is a one-seed screen, not final evidence. Its purpose is to eliminate clearly inefficient architectures and identify two finalists.

In [ ]:
RUN_SCREEN = True
screen_runs = []

if RUN_SCREEN:
    for label, hidden_sizes in ADVANTAGE_ARCHITECTURES.items():
        print(f'\n=== {label} ===')
        run = train_for_fixed_time(
            label,
            hidden_sizes,
            seed=SEED,
            training_seconds=SCREEN_MINUTES_PER_ARCHITECTURE * 60,
        )
        screen_runs.append(run)
        gc.collect()
        torch.cuda.empty_cache()

len(screen_runs)

In [ ]:
def exploitability(point, key):
    return 2 * point[key] - 1.0


def normalized_auc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or x[-1] <= x[0]:
        return np.nan
    return float(np.trapezoid(y, x) / (x[-1] - x[0]))


summary_rows = []
evaluation_rows = []
for run in screen_runs:
    evaluations = run['evaluations']
    training = run['training_records']
    evaluation_rows.extend(evaluations)
    x = [point['measured_training_s'] for point in evaluations]
    exact = [exploitability(point, 'exact_average_predicted_avg') for point in evaluations]
    learned = [exploitability(point, 'predicted_avg') for point in evaluations]
    current = [exploitability(point, 'current_predicted_avg') for point in evaluations]
    final_training = training[-1]
    summary_rows.append({
        'configuration': run['configuration'],
        'advantage parameters': run['parameters_per_advantage_network'],
        'strategy parameters': run['parameters_per_strategy_network'],
        'iterations completed': run['iterations'],
        'final exact-average exploitability': exact[-1],
        'best exact-average exploitability': min(exact),
        'exact-average normalized AUC': normalized_auc(x, exact),
        'final learned-average exploitability': learned[-1],
        'final learned-minus-exact gap': learned[-1] - exact[-1],
        'final current exploitability': current[-1],
        'final held-out advantage MSE': evaluations[-1]['advantage_validation_mse'],
        'final held-out strategy TV': evaluations[-1]['advantage_validation_strategy_tv'],
        'mean traversal s': float(np.mean([
            row['timing']['traversal_s'] for row in training
        ])),
        'mean advantage fit s': float(np.mean([
            row['timing']['advantage_training_s'] for row in training
        ])),
        'mean strategy fit s': float(np.mean([
            row['timing']['strategy_training_s'] for row in training
        ])),
        'advantage records seen': sum(final_training['advantage_records_seen']),
        'strategy records seen': sum(final_training['strategy_records_seen']),
    })

summary_df = pd.DataFrame(summary_rows).set_index('configuration').sort_values(
    ['exact-average normalized AUC', 'final exact-average exploitability']
)
evaluation_df = pd.DataFrame(evaluation_rows)
display(summary_df.style.format(precision=6).background_gradient(
    subset=[
        'final exact-average exploitability',
        'best exact-average exploitability',
        'exact-average normalized AUC',
    ],
    cmap='RdYlGn_r',
))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for label, group in evaluation_df.groupby('configuration', sort=False):
    x = group['measured_training_s'] / 60.0
    axes[0, 0].plot(x, 2 * group['exact_average_predicted_avg'] - 1, marker='o', label=label)
    axes[0, 1].plot(x, 2 * group['predicted_avg'] - 1, marker='o', label=label)
    axes[0, 2].plot(x, 2 * group['current_predicted_avg'] - 1, marker='o', label=label)
    axes[1, 0].plot(x, group['advantage_validation_mse'], marker='o', label=label)
    axes[1, 1].plot(x, group['advantage_validation_strategy_tv'], marker='o', label=label)

fit = summary_df.reset_index()
axes[1, 2].scatter(
    fit['mean advantage fit s'],
    fit['final exact-average exploitability'],
    s=80,
)
for _, row in fit.iterrows():
    axes[1, 2].annotate(
        row['configuration'],
        (row['mean advantage fit s'], row['final exact-average exploitability']),
    )

plot_specs = [
    ('Exact generated average', 'Exploitability', True),
    ('Learned average', 'Exploitability', True),
    ('Current strategy', 'Exploitability', True),
    ('Held-out advantage MSE', 'MSE', False),
    ('Held-out induced-strategy TV', 'TV', False),
    ('Capacity versus final quality', 'Final exact-average exploitability', False),
]
for ax, (title, ylabel, log_y) in zip(axes.flat, plot_specs):
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, which='both', alpha=0.3)
    if log_y:
        ax.set_yscale('log')
for ax in axes[0, :]:
    ax.set_xlabel('Measured training minutes')
for ax in axes[1, :2]:
    ax.set_xlabel('Measured training minutes')
axes[1, 2].set_xlabel('Mean advantage fitting seconds per iteration')
axes[0, 0].legend(fontsize=8)
fig.tight_layout();

## Optional confirmation

After inspecting the screen, enter the two finalist labels below. The confirmation uses three seeds and a longer equal-time budget. Leave `RUN_CONFIRMATION = False` until the finalists are selected.

In [ ]:
RUN_CONFIRMATION = False
FINALIST_LABELS = ('1024x1024', '1024x1024x1024x1024')
CONFIRMATION_SEEDS = (7, 17, 27)
CONFIRMATION_MINUTES_PER_RUN = 15

confirmation_runs = []
if RUN_CONFIRMATION:
    for seed in CONFIRMATION_SEEDS:
        for label in FINALIST_LABELS:
            print(f'\n=== seed={seed}; {label} ===')
            run = train_for_fixed_time(
                label,
                ADVANTAGE_ARCHITECTURES[label],
                seed=seed,
                training_seconds=CONFIRMATION_MINUTES_PER_RUN * 60,
            )
            run['seed'] = seed
            confirmation_runs.append(run)
            gc.collect()
            torch.cuda.empty_cache()

confirmation_rows = []
for run in confirmation_runs:
    evaluations = run['evaluations']
    x = [point['measured_training_s'] for point in evaluations]
    exact = [exploitability(point, 'exact_average_predicted_avg') for point in evaluations]
    confirmation_rows.append({
        'seed': run['seed'],
        'configuration': run['configuration'],
        'iterations completed': run['iterations'],
        'final exact-average exploitability': exact[-1],
        'best exact-average exploitability': min(exact),
        'exact-average normalized AUC': normalized_auc(x, exact),
    })

confirmation_df = pd.DataFrame(confirmation_rows)
if not confirmation_df.empty:
    display(confirmation_df.set_index(['seed', 'configuration']).sort_index())
    display(confirmation_df.groupby('configuration').agg(['mean', 'std']))